In [10]:
import pandas as pd
from dgilit import PubMedClient
import http.client
import os
import urllib.error
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from tqdm import tqdm

from dotenv import load_dotenv
import os

load_dotenv()

MC_EMAIL = os.environ["MC_EMAIL"]



# Grab Text
Now that PMIDs have been identified within 5 years and qualified, grab those that are just within a certain NIH percentile and grab and save the text

In [11]:
df = pd.read_csv('../../../data/2026-07-06-test03-pmid-statistics.csv').drop(labels='Unnamed: 0',axis=1)
df.head()

,pmid,publication_year,title,journal,citation_count,rcr,nih_percentile,sjr,h_index,citations_per_year
0,29799308,2021,Genetic Influences on Patient-Oriented Outcome...,Journal of neurotrauma,57.0,4.500048,90.9,1.329,187.0,9.500000
1,29927639,2021,Retracted: Microcalcification-associated breas...,The British journal of radiology,8.0,0.412753,24.6,NaN,NaN,1.333333
2,30199582,2024,WITHDRAWN: Glucocorticoids ameliorate periosti...,Clinical and experimental allergy : journal of...,19.0,4.995387,92.3,NaN,NaN,6.333333
3,30287320,2023,Association of multiple primary melanomas with...,Journal of the American Academy of Dermatology,3.0,0.972763,49.4,1.344,270.0,0.750000
4,30361487,2021,Genome-wide association study of brain amyloid...,Molecular psychiatry,58.0,3.314019,85.9,4.602,274.0,9.666667


In [12]:
df = (
    df[df["nih_percentile"] > 99]
    .sort_values(
        ["nih_percentile", "rcr", "citation_count"],
        ascending=[False, False, False]
    )
)

df

,pmid,publication_year,title,journal,citation_count,rcr,nih_percentile,sjr,h_index,citations_per_year
9854,33495651,2021,"Ferroptosis: mechanisms, biology and role in d...",Nature reviews. Molecular cell biology,6212.0,435.276004,100.0,NaN,NaN,1035.333333
13720,33597522,2021,Inference and analysis of cell-cell communicat...,Nature communications,6337.0,386.627302,100.0,4.904,634.0,1056.166667
4766,33277608,2021,Engineering precision nanoparticles for drug d...,Nature reviews. Drug discovery,4882.0,365.848617,100.0,NaN,NaN,813.666667
112002,35732831,2022,The 5th edition of the World Health Organizati...,Leukemia,3330.0,305.373398,100.0,3.675,235.0,666.000000
112001,35732829,2022,The 5th edition of the World Health Organizati...,Leukemia,3101.0,296.739071,100.0,3.675,235.0,620.200000
...,...,...,...,...,...,...,...,...,...,...
209126,37815057,2023,Phenotypic Switching of Vascular Smooth Muscle...,Journal of the American Heart Association,151.0,17.456656,99.1,2.030,167.0,37.750000
107330,35630768,2022,A Comprehensive Review of Rosmarinic Acid: Fro...,"Molecules (Basel, Switzerland)",135.0,17.454059,99.1,NaN,NaN,27.000000
130523,36109501,2022,Exosomes and cancer - Diagnostic and prognosti...,Oncogenesis,196.0,17.451699,99.1,2.081,78.0,39.200000
167264,36860361,2023,Exosomes as natural nanocarrier-based drug del...,3 Biotech,125.0,17.451136,99.1,0.658,102.0,31.250000


In [13]:
def join_unique(values: pd.Series) -> str:
    return "; ".join(sorted({str(value) for value in values.dropna() if str(value)}))


FETCH_ERRORS = []
FETCH_ERROR_TYPES = (
    ET.ParseError,
    http.client.IncompleteRead,
    TimeoutError,
    urllib.error.HTTPError,
    urllib.error.URLError,
)


def record_fetch_error(stage: str, pmids: list[str], error: Exception) -> None:
    FETCH_ERRORS.append({
        "stage": stage,
        "pmids": ";".join(map(str, pmids)),
        "pmid_count": len(pmids),
        "error_type": type(error).__name__,
        "error_message": str(error),
    })

def fetch_articles_with_progress(
    client: PubMedClient,
    pmids: list[str],
    *,
    chunk_size: int = 100,
    include_full_text: bool = True,
    desc: str = "Fetching PubMed/PMC article chunks",
):
    articles = {}
    stage = "full_text" if include_full_text else "metadata"

    for start in tqdm(
        range(0, len(pmids), chunk_size),
        total=(len(pmids) + chunk_size - 1) // chunk_size,
        desc=desc,
    ):
        chunk = pmids[start:start + chunk_size]
        try:
            articles.update(
                client.fetch_articles(
                    chunk,
                    include_full_text=include_full_text,
                )
            )
            continue
        except FETCH_ERROR_TYPES as error:
            record_fetch_error(stage=f"{stage}_chunk", pmids=chunk, error=error)
        except Exception as error:
            record_fetch_error(stage=f"{stage}_chunk_unexpected", pmids=chunk, error=error)

        if len(chunk) == 1:
            continue

        for pmid in tqdm(chunk, desc=f"Retrying failed {stage} chunk one PMID at a time", leave=False):
            try:
                articles.update(
                    client.fetch_articles(
                        [pmid],
                        include_full_text=include_full_text,
                    )
                )
            except FETCH_ERROR_TYPES as error:
                record_fetch_error(stage=f"{stage}_pmid", pmids=[pmid], error=error)
            except Exception as error:
                record_fetch_error(stage=f"{stage}_pmid_unexpected", pmids=[pmid], error=error)

    return articles

client = PubMedClient(
    email=MC_EMAIL,
    batch_size=25,
)

pmids = df['pmid'].to_list()

articles = fetch_articles_with_progress(
    client,
    pmids,
    chunk_size=25,
    include_full_text=True,
    desc="Fetching recent PubMed/PMC article chunks",
)

Fetching recent PubMed/PMC article chunks: 100%|██████████| 86/86 [04:51<00:00,  3.38s/it]


In [26]:
from pathlib import Path
import pickle

def save_articles(articles: dict[str, "PubMedArticle"], filename: str | Path) -> None:
    """
    Save a dictionary of PubMedArticle objects to disk.

    Parameters
    ----------
    articles : dict
        Dictionary in the format:
        {
            "12345678": PubMedArticle(...),
            ...
        }
    filename : str or Path
        Output file (e.g. "articles.pkl")
    """
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)

    with open(filename, "wb") as f:
        pickle.dump(articles, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_articles(filename: str | Path) -> dict[str, "PubMedArticle"]:
    """
    Load a previously saved PubMedArticle dictionary.
    """
    with open(filename, "rb") as f:
        return pickle.load(f)

In [28]:
# Save
save_articles(articles, "../../../data/2026-07-07-test03-articles.pkl")

# Later...
pubmed_articles = load_articles("../../../data/2026-07-07-test03-articles.pkl")

print(len(pubmed_articles))


2148


In [18]:
dir(articles['33495651'])

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__',
 '__pydantic_serializer__',
 '__py

In [24]:
# Articles have sections (chunks of text)
for section in articles['33495651'].sections:
    print(section.section_id)

ABS1
S1
S2
S3
S4
S5
S6
S7
S8
S9
S10
S11
S12
S13
S14
S15
S16
S17
S18
S19
S20
S21
S22
S23
